# NVML：用 Python 读 GPU 状态

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-06-nvml.ipynb)

直接对接 HAMi 源码。用 pynvml 调用 NVML 接口，拿到温度、功耗、显存、利用率、UUID、MIG 能力，这些正是 HAMi 设备发现与配额管理依赖的同一套 API。

本 notebook 配合教程章节 [NVML：用 Python 读 GPU 状态](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

前面五个 Lab 你一直用 `nvidia-smi` 命令行看 GPU。但在真实平台里，HAMi、DCGM、Prometheus、Kubernetes device plugin 这些组件都需要**程序化地**读 GPU 状态，而它们用的都是同一套底层：**NVML**。本 Lab 让你亲手调 NVML 的 Python 绑定 `pynvml`，拿到温度、功耗、显存、利用率、UUID，再把这些字段和 `nvidia-smi` 的输出、和 HAMi 源码里的调用一一对应。做完这一章，你再看 HAMi 的 device-plugin 代码就不会陌生。

## 第 1 步：安装并初始化

In [ ]:
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
print("NVML version  :", nvmlSystemGetNVMLVersion())
print("driver version:", nvmlSystemGetDriverVersion())

> **装 nvidia-ml-py，不要装 pynvml**
>
> PyPI 上的 `pynvml` 包已被 NVIDIA 标记为废弃，官方维护的是 **`nvidia-ml-py`**，两者都提供 `import pynvml` 这个模块名。新版的字符串返回值已经是 `str`，不再是 `bytes`。HAMi 的 Go 代码用的是同一套 NVML C API 的 Go 绑定 `go-nvml`。

## 第 2 步：拿到设备句柄和身份

In [ ]:
handle = nvmlDeviceGetHandleByIndex(0)
print("name :", nvmlDeviceGetName(handle))
print("uuid :", nvmlDeviceGetUUID(handle))
print("index:", nvmlDeviceGetIndex(handle))

预期：`name: Tesla T4`、`uuid: GPU-xxxxxxxx-...`。

> **为什么 UUID 这么重要**
>
> HAMi 在调度时识别 GPU 的核心字段就是 **UUID**。当多个 Pod 共享一张卡时，HAMi 在 Pod 注解里写的就是 `{UUID},{vendor},{VRAM},{compute%}`（见 [L1 HAMi 第一份共享作业](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/single-gpu/hami-first-share/)）。你现在拿到的这个 `GPU-...`，就是那张被切分/共享的卡的唯一身份。

## 第 3 步：显存信息

In [ ]:
mem = nvmlDeviceGetMemoryInfo(handle)
print("total MiB :", round(mem.total/1024**2))
print("used  MiB :", round(mem.used/1024**2))
print("free  MiB :", round(mem.free/1024**2))

和 `nvidia-smi --query-gpu=memory.used` 的口径**完全一致**：[Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 里你看到的“caching allocator 攥着显存不还”现象，用 NVML 读出来同样存在。HAMi 的 `nvidia.com/gpumem` 配额正是基于这层口径。

## 第 4 步：利用率与功耗、温度

In [ ]:
util = nvmlDeviceGetUtilizationRates(handle)
print("gpu util % :", util.gpu)
print("mem util % :", util.memory)   # 注意：这是显存控制器利用率，不是占用率

print("power W    :", nvmlDeviceGetPowerUsage(handle)/1000.0, "/", nvmlDeviceGetPowerManagementLimit(handle)/1000.0)
print("temp C     :", nvmlDeviceGetTemperature(handle, NVML_TEMPERATURE_GPU))
print("perf state :", nvmlDeviceGetPerformanceState(handle))   # P8 空载 / P0 满载
try:
    print("fan %      :", nvmlDeviceGetFanSpeed(handle))
except NVMLError_NotSupported:
    print("fan %      : not supported (T4 是被动散热，没有风扇)")

`NVMLError_NotSupported` 是你读 HAMi / DCGM 代码时会反复遇到的异常：同一套 API 要覆盖从消费卡到数据中心卡的所有产品，很多字段在某些卡上就是不存在的。

注意区分两个“显存”指标：

- `memory.used` / `total`（第 3 步）：**显存占用**（字节）；
- `utilization.memory`（第 4 步）：**显存控制器忙闲比**（百分比）。

后者是 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) “纯拷贝时 GPU-Util 低、但显存其实很忙”的那个“忙”，它和 GPU-Util 是两个独立维度。

## 第 5 步：谁在用这张卡

In [ ]:
import torch; torch.zeros(1, device="cuda")   # 让当前进程持有一个 CUDA context
procs = nvmlDeviceGetComputeRunningProcesses(handle)
print("compute processes:", len(procs))
for p in procs:
    print("  pid", p.pid, "used MiB", None if p.usedGpuMemory is None else round(p.usedGpuMemory/1024**2))

这是 HAMi 和 DCGM 用来回答“哪个容器在用哪张卡、用了多少”的接口。在 Colab 这样的容器里你可能会看到 PID 和宿主机不一致、甚至列表为空：因为 NVML 返回的是**宿主机 PID 命名空间**的号码。这个“容器里看不到自己的 GPU 进程”问题，正是 Kubernetes 上 GPU 监控要靠 DCGM Exporter 做 Pod 归属映射的原因。

## 第 6 步：MIG 能力探测（为 [Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 铺垫）

In [ ]:
try:
    mode, pending = nvmlDeviceGetMigMode(handle)
    print("MIG mode:", mode, "pending:", pending)
except NVMLError_NotSupported:
    print("MIG not supported on this GPU")

T4 不支持 MIG（需要 Ampere 及以上的 A100 / A30 / H100 等）。你会看到 `not supported`，这本身是个有用的事实：**不是所有卡都能做硬隔离 MIG**，这正是 [Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 里“为什么要有 MPS / time-slicing”的起点。

## 验证

完成后你应该能：

- 用 `pynvml` 拿到 name / uuid / memory / util / power / temp；
- 区分“显存占用”和“显存控制器利用率”；
- 解释 UUID 在 HAMi 共享调度里的角色；
- 知道 T4 为何查不到 MIG mode。

## 故障模式

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `nvmlInit()` 报 `Driver not loaded` | runtime 是 CPU | 切到 T4 |
| `nvmlDeviceGetFanSpeed` 抛 `NVMLError_NotSupported` | T4 无风扇 | try/except 包住（代码已处理） |
| 字符串字段返回 `b'...'` | 装到了老版 `pynvml` | `pip install -U nvidia-ml-py` |
| `GetComputeRunningProcesses` 返回空 | 容器 PID 命名空间隔离 | 正常，记录为“容器内不可见” |
| MIG 调用报 `Not Supported` | T4 架构不支持 | 正常，记录为“不支持” |

## 记录你的实验元数据

运行完成后，把实际数值回填到教程该章的实验元数据表：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的故障模式。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。